# Intentlock: Phases 1-2: testbed and dataset (no GPU needed)

### Setup
Runtime → Change runtime type → **T4 GPU** (or better). Accept the Tiny Aya licence at https://huggingface.co/CohereLabs/tiny-aya-global first: the weights are gated.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: switch the runtime to a GPU'

In [ ]:
import os
if not os.path.exists('intentlock'):
    !git clone -q https://github.com/davieslennox0/intentlock.git
%cd intentlock
!pip uninstall -y -q torchao            # SafeTune's Colab note
!pip install -q -r requirements.txt 2>&1 | tail -3
!pip check | head -20

In [ ]:
from huggingface_hub import login, whoami
login()        # token with read access to Tiny Aya and write access to your namespace
HF_USER = whoami()['name']; print(HF_USER)

In [ ]:
!python -m pytest -q tests

Regenerate and curate. The committed `data/curated/` files should be reproduced exactly.

In [ ]:
!python -m data.generate --seed 7
!python -m data.curate | tail -45

In [ ]:
import json, collections
rows = [json.loads(l) for s in ('train', 'test', 'heldout_format') for l in open(f'data/curated/{s}.jsonl')]
for k, v in sorted(collections.Counter((r['family'], r['split']) for r in rows).items()): print(k, v)

Five hand-written scenarios through the base agent, with full traces.

In [ ]:
!python -m testbed.runner --scenarios data/seed/handwritten.jsonl --out runs/hand_base.jsonl --config-name base

In [ ]:
import json
for l in open('runs/hand_base.jsonl'):
    t = json.loads(l)
    print('=' * 80); print(t['scenario_id'], t['family'], '| labels:', t['labels'])
    for s in t['steps']: print('  ', s.get('tool') or 'FINAL', json.dumps(s.get('args') or s.get('final') or s.get('raw'))[:200])

In [ ]:
!python scripts/publish_hf.py --user {HF_USER} --what dataset